# Learn Marginal Rate of Substitution between UM171 and UM729 for Erythroid Progenitor Enrichment.

## Prepare the Notebook.

### Flags for Autoreload.

In [ ]:
%reload_ext autoreload
%autoreload 2

### Import Libraries.

In [ ]:
from functools import partial
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch


import hydra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import torch
from tqdm import tqdm

import sc_exp_design

sys.path.insert(0, "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/shared_utils")

from experiment_utils import (
    get_forward_model, generate_with_condition, generated_density_knn, 
    get_adata_from_idx, get_loss_fn, get_target_dict
)
from data_utils import (
    standardize_array_and_write_to_adata, get_condition_data_from_file,
    get_protocol_tranformations, get_target_adata_marker_opt
)
from inverse_utils import (
    map_df, l2_loss, l1_loss, hinge_loss, cauchy_loss, loss_fn_factory_marker_opt,
    linear_scheduler_with_warmup, constraint_fn_factory, loss_fn_factory, map_df

)
from lambda_schedulers import ReciprocalDecayScheduler
from plot_utils import plot_all_markers_overlay


### Initialize Configurations.

In [ ]:
with hydra.initialize(
    config_path="../../../../../inverse/loss_guidance/config"
):
    config = hydra.compose(
        config_name="run_inverse_constrained",
        overrides=[
            "paths=bloodplus_fm",
            "annotation=bloodplus",
            "sampling.target_cell_type=EryPro",
            "sampling.N=10",
            "sampling.num_time_steps=20",
            "forward_model.num_time_steps=10",
            "loss_guidance.num_forward_pass_per_sample=100",
            "constraints=default_all_cols"
        ]
    )
protocol_columns = list(config.annotation.protocol_columns)

### Define Utility Functions.

In [ ]:
def compute_mrs_learning_loss(
    loss_fn,
    x_batch,
    resc_module,
    protocol_columns,
    input_mol="um171_[nm]",
    output_mol="um729_[nm]",
    compute_ct_loss=False,
    loss_type="bce",
    lambda_mrs=1.0,
):
    def inv_resc_module(Xj):
        if resc_module.bias is not None:
            return (Xj - resc_module.bias)/resc_module.weight
        return Xj/resc_module.weight

    def _compute_mrs_loss(x_resc, X):
        # 4. Forward through your model
        F_vals = loss_fn(x_resc)   # (batch,)
        if lambda_mrs == 0.0:
            return 0.0, F_vals
        # 5. Compute total derivative dF / d(input_val)
        #    Because X_j = h(X_i), this total derivative is exactly ∂F/∂X_i + ∂F/∂X_j * h'(X_i)
        dF_dinput, = torch.autograd.grad(F_vals.sum(), X, create_graph=True)

        # 6. Force it to zero
        return (dF_dinput ** 2).mean(), F_vals

    def _compute_ct_loss(p_orig, p_resc):
        if loss_type == 'mse':
            return ((p_orig - p_resc) ** 2).mean()
        elif loss_type == 'bce':
            # binary cross-entropy: numerical stability version
            eps = 1e-7
            p_resc_clamped = p_resc.clamp(eps, 1.0 - eps)
            return - (p_orig * torch.log(p_resc_clamped) +
                                (1-p_orig) * torch.log(1-p_resc_clamped)).mean()
        else:
            raise ValueError

    # 0. retrieve indices 
    input_idx = protocol_columns.index(input_mol)
    output_idx = protocol_columns.index(output_mol)

    # 1. Independent variable with gradient tracking
    Xi = x_batch[..., input_idx].clone().detach().requires_grad_(True).unsqueeze(-1)
    Xj = x_batch[..., output_idx].clone().detach().requires_grad_(True).unsqueeze(-1)

    # 2. Predicted output molecule (graph attached)
    Xj_hat = resc_module(Xi)  # shape (batch, 1)
    Xi_hat = inv_resc_module(Xj)  # shape (batch, 1)

    # 3. Build rescaled input – must keep gradient flow from output_val (conversion i -> j)
    x_resc_ij = x_batch.clone()
    x_resc_ij[..., output_idx] = Xj_hat.squeeze(-1)  # shape (batch,)

    # 4. Build rescaled input – must keep gradient flow from output_val (conversion j -> i)
    x_resc_ji = x_batch.clone()
    x_resc_ji[..., input_idx] = Xi_hat.squeeze(-1)  # shape (batch,)

    # 5. compute losses and sum up
    loss_ij, F_vals_ij = _compute_mrs_loss(x_resc_ij, Xi)
    loss_ji, F_vals_ji = _compute_mrs_loss(x_resc_ji, Xj)
    loss_mrs = loss_ij + loss_ji

    # 6. compute consistency losses
    if compute_ct_loss:
        F_vals = loss_fn(x_batch)
        p_orig = torch.exp(-F_vals)
        p_ij = torch.exp(-F_vals_ij)
        p_ji = torch.exp(-F_vals_ji)
        ct_loss_ij = _compute_ct_loss(p_orig, p_ij)
        ct_loss_ji = _compute_ct_loss(p_orig, p_ji)
        ct_loss = ct_loss_ij + ct_loss_ji
    else:
        ct_loss = torch.zeros_like(loss_mrs)
    return lambda_mrs*loss_mrs + ct_loss


In [ ]:
# def compute_mrs_learning_loss(
#     loss_fn,               # returns -log(prob_target_class) per sample
#     x_batch,
#     resc_module,
#     protocol_columns,
#     input_mol="um171_[nm]",
#     output_mol="um729_[nm]",
#     loss_type="mse",       # 'mse' or 'smooth_l1'
# ):
#     # Inverse rescaling
#     def inv_resc_module(Xj):
#         w = resc_module.weight
#         b = resc_module.bias
#         if b is not None:
#             return (Xj - b) / w
#         return Xj / w

#     input_idx = protocol_columns.index(input_mol)
#     output_idx = protocol_columns.index(output_mol)

#     # Original loss value (detached – fixed target)
#     with torch.no_grad():
#         L_orig = loss_fn(x_batch)                # (batch,)

#     # Forward direction i -> j
#     Xi = x_batch[..., input_idx].unsqueeze(-1)
#     Xj_hat = resc_module(Xi).squeeze(-1)
#     x_resc_ij = x_batch.clone()
#     x_resc_ij[..., output_idx] = Xj_hat
#     L_ij = loss_fn(x_resc_ij)                    # (batch,)

#     # Inverse direction j -> i
#     Xj = x_batch[..., output_idx].unsqueeze(-1)
#     Xi_hat = inv_resc_module(Xj).squeeze(-1)
#     x_resc_ji = x_batch.clone()
#     x_resc_ji[..., input_idx] = Xi_hat
#     L_ji = loss_fn(x_resc_ji)                    # (batch,)

#     # Consistency loss on the loss values
#     if loss_type == "mse":
#         loss_fwd = ((L_orig - L_ij) ** 2).mean()
#         loss_inv = ((L_orig - L_ji) ** 2).mean()
#     elif loss_type == "smooth_l1":
#         loss_fwd = torch.nn.functional.smooth_l1_loss(L_ij, L_orig)
#         loss_inv = torch.nn.functional.smooth_l1_loss(L_ji, L_orig)
#     else:
#         raise ValueError

#     return loss_fwd + loss_inv

In [ ]:
def compute_mrs_learning_loss(loss_fn, x_batch, resc_module, protocol_columns,
                              input_mol, output_mol):
    # Inverse (handles bias / no bias)
    def inv_resc(Xj):
        w = resc_module.weight
        b = resc_module.bias
        return Xj / w if b is None else (Xj - b) / w

    inp_idx = protocol_columns.index(input_mol)
    out_idx = protocol_columns.index(output_mol)

    # Original log‑prob (detached, fixed target)
    with torch.no_grad():
        L_orig = loss_fn(x_batch)                     # (batch,)

    # Forward i→j
    Xi = x_batch[..., inp_idx].unsqueeze(-1)
    Xj_hat = resc_module(Xi).squeeze(-1)
    x_ij = x_batch.clone()
    x_ij[..., out_idx] = Xj_hat
    L_ij = loss_fn(x_ij)

    # Inverse j→i
    Xj = x_batch[..., out_idx].unsqueeze(-1)
    Xi_hat = inv_resc(Xj).squeeze(-1)
    x_ji = x_batch.clone()
    x_ji[..., inp_idx] = Xi_hat
    L_ji = loss_fn(x_ji)

    # Robust consistency – smooth L1 (Huber) is far less jumpy than MSE
    loss_fwd = torch.nn.functional.smooth_l1_loss(L_ij, L_orig)
    loss_inv = torch.nn.functional.smooth_l1_loss(L_ji, L_orig)

    return loss_fwd + loss_inv

## Load Stuff.

### Load Forward Model.

In [ ]:
forward_model, (
    phi_model,
    target_prediction_model
) = get_forward_model(config)
forward_model.target_prediction_model.resc_model["model"].eval()
forward_model.forward_model.velocity_field.eval()
print(forward_model.target_prediction_model.resc_model["model"])
print(forward_model.forward_model.velocity_field)

### Load Solutions for Inverse Model.

In [ ]:
load_filtered = False

target_ct = "EryPro"
if load_filtered:
    candidates_path = f"/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/manually_filtered/original_bloodplus/filtered/filtered_candidates_{target_ct}.csv"
    main_df = pd.read_csv(candidates_path)
else:

    base_results_dir = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance"

    session_name = "bloodplus_inverse_fm_new_forward" 
    optim_type = "unconstrained-pure_populations-reciprocal"

    ct_run_dir = os.path.join(
        base_results_dir,
        session_name,
        optim_type,
        target_ct
    )

    all_dfs_list = []
    runs = os.listdir(ct_run_dir)
    for run in runs:
        run_dir = os.path.join(ct_run_dir, run)
        array_path = os.path.join(run_dir, "candidates.csv")
        run_df = pd.read_csv(array_path)
        all_dfs_list.append(run_df)
    main_df =  pd.concat(all_dfs_list, axis=0)
main_df


## MRS Inference.

### Get transformed values.

In [ ]:
if load_filtered:
    # get column df
    cond_df = main_df[protocol_columns]

    # get transformation

    trans_cond_df = pd.DataFrame(
        np.log1p(cond_df.values),
        columns=protocol_columns
    )
    # trans_cond_df = main_df.map(lambda e: np.log1p)

    # prepare condition tensor
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float32
    X_cond = torch.from_numpy(trans_cond_df.values).to(dtype).to(device)
    print(f"{X_cond.shape=}")
else:
    # get column names
    transformed_protocol_columns = [f"{col}:rescaled" for col in protocol_columns]

    # get column df
    cond_df = main_df[protocol_columns]
    trans_cond_df = main_df[transformed_protocol_columns]

    # prepare condition tensor
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float32
    X_cond = torch.from_numpy(trans_cond_df.values).to(dtype).to(device)
    print(f"{X_cond.shape=}")

### Fit Label Encoder.

In [ ]:
ct_le = LabelEncoder()
ct_values = target_prediction_model.train_data.adata.obs["cell_type"].values
ct_le.fit(ct_values)
classes = ct_le.classes_.tolist()

### Get Target.

In [ ]:
config.sampling.target_cell_type = target_ct
target = get_target_dict(
    config,
    classes,
    forward_model.forward_model.device
)

### Instantiating Loss Function.

In [ ]:
batch_size = 8
config.sampling.N = batch_size
config.sampling.target_cell_type = target_ct
config.sampling.mask = [True]*len(classes)


non_linearity = torch.nn.Identity()
loss_fns = get_loss_fn(config)
compute_loss, noise = loss_fn_factory(
    loss_fns,
    config,
    target,
    non_linearity,
    forward_model,
)

### Learn Rescaling.

In [ ]:
# set reproducibility
random_seed = 42
sc_exp_design.utils.set_reproducibility(random_seed)

# initialize model
bias = True
use_xavier_init = True
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
resc_module = torch.nn.Linear(1, 1, bias=bias).to(dtype).to(device)
if use_xavier_init:
    torch.nn.init.xavier_uniform_(resc_module.weight)

# input and output molecules
input_mol = "um171_[nm]"
output_mol = "um729_[µm]"

# settings for training
n_steps = 350
lr = 5e-4
optim = torch.optim.Adam(resc_module.parameters(), lr=lr)

# store for loss
loss_history = []

# iterate over training steps
iterator = range(1, n_steps + 1)
pbar = tqdm(iterator)
for step in iterator:
    # get batch
    batch_idx = np.random.choice(X_cond.shape[0], size=batch_size)
    x_batch = X_cond[batch_idx]

    # compute loss
    mrs_loss = compute_mrs_learning_loss(
        compute_loss,
        x_batch,
        resc_module,
        protocol_columns=protocol_columns,
        input_mol=input_mol,
        output_mol=output_mol,
    )

    # backprop
    mrs_loss.backward()
    optim.step()

    # update progress bar
    mrs_loss_val = mrs_loss.item()
    loss_history.append(mrs_loss.item())
    pbar.set_description(f"MRS Loss {step=}, {mrs_loss_val=}")
    pbar.update()
plt.plot(loss_history)


### Check Total Derivative.

In [ ]:
# def evaluate_conversion(
#     loss_fn, X_cond, resc_module, protocol_columns,
#     input_mol="um171_[nm]", output_mol="um729_[µm]",
#     batch_size=32
# ):
#     input_idx = protocol_columns.index(input_mol)
#     output_idx = protocol_columns.index(output_mol)
#     n = X_cond.shape[0]

#     all_dF = []
#     all_F_vals = []
#     all_F_orig_vals = []   # if you want original F for comparison, optional

#     for start in range(0, n, batch_size):
#         end = min(start + batch_size, n)
#         x_batch = X_cond[start:end]

#         if x_batch.shape[0] != batch_size:
#             n_obs_orig = x_batch.shape[0]
#             x_batch_orig = x_batch.clone()

#             x_batch = torch.zeros((batch_size, x_batch_orig.shape[-1]), device=x_batch_orig.device, dtype=x_batch_orig.dtype)
#             x_batch[:n_obs_orig] = x_batch
#         else:
#             n_obs_orig = None

#         input_val = x_batch[:, input_idx].clone().detach().requires_grad_(True).unsqueeze(-1)
#         output_val = resc_module(input_val)
#         x_resc = x_batch.clone()
#         x_resc[:, output_idx] = output_val.squeeze(-1)

#         # F for rescaled
#         F_resc = loss_fn(x_resc)
#         if n_obs_orig is not None:
#             F_resc = F_resc[:n_obs_orig]

#         # total derivative dF/dinput_val
#         dF = torch.autograd.grad(F_resc.sum(), input_val, create_graph=False)[0]  # (batch,1)
#         all_dF.append(dF.detach().abs().cpu().numpy())
#         all_F_vals.append(F_resc.detach().cpu().numpy())

#     # Concatenate
#     dF_abs = np.concatenate(all_dF, axis=0).flatten()      # (N,) absolute values
#     F_vals_arr = np.concatenate(all_F_vals, axis=0)        # (N,)
#     mean_abs_dF = dF_abs.mean()
#     std_F = F_vals_arr.std()   # population std across all conditions

#     print(f"Mean |dF/dX_i| across all conditions: {mean_abs_dF:.6f}")
#     print(f"Std of F across conditions: {std_F:.6f}")

#     # To create scatter: need orig_df and resc_df (per-condition probabilities)
#     # We'll reuse the existing orig_df and resc_df you already have,
#     # or recompute them similarly in a batched way (not shown here).
#     # Assume you have orig_df and resc_df DataFrames already.
#     return F_vals_arr  # optionally return for further analysis
# F_vals_arr = evaluate_conversion(compute_loss, X_cond, resc_module, protocol_columns)

## Try Rescaling.

### Rescale Solutions.

In [ ]:
def inv_resc_module(Xj):
    return (Xj - resc_module.bias)/resc_module.weight

# 0. retrieve indices 
input_idx = protocol_columns.index(input_mol)
output_idx = protocol_columns.index(output_mol)

# 1. Independent variable with gradient tracking
Xi = X_cond[..., input_idx].clone().detach().requires_grad_(True).unsqueeze(-1)
Xj = X_cond[..., output_idx].clone().detach().requires_grad_(True).unsqueeze(-1)

# 2. Predicted output molecule (graph attached)
Xj_hat = resc_module(Xi)  # shape (batch, 1)
Xi_hat = resc_module(Xj)  # shape (batch, 1)

# 3. Build rescaled input – must keep gradient flow from output_val
X_cond_resc_ij = X_cond.clone()
X_cond_resc_ij[..., output_idx] = Xj_hat.squeeze(-1)  # shape (batch,)

X_cond_resc_ji = X_cond.clone()
X_cond_resc_ji[..., input_idx] = Xi_hat.squeeze(-1)  # shape (batch,)

### Query Forward Model with Original Protocols.

In [ ]:
num_time_steps = 10
solver_kwargs = {"method": "euler", "atol": 1e-5, "rtol": 1e-5}
chuck_size = 16

all_ct_probs = []
for i in tqdm(range(0, X_cond.shape[0], chuck_size)):
    x_cond = X_cond[i:i+chuck_size]
    fwd_out_orig = generate_with_condition(
        x_cond.detach().cpu().numpy(),
        forward_model,
        num_time_steps,
        solver_kwargs,
        ct_le,
        num_samples=350,
        noise=None,
        n_scatter_feats=6,
    )
    ct_probs = fwd_out_orig["ct_probs"]
    all_ct_probs.append(ct_probs)
all_ct_probs = np.concatenate(all_ct_probs, axis=1).mean(0)
orig_df = pd.DataFrame(all_ct_probs, columns=classes)
orig_df

### Query Forward Model with Rescaled Protocol (i -> j).

In [ ]:
num_time_steps = 10
solver_kwargs = {"method": "euler", "atol": 1e-5, "rtol": 1e-5}
chuck_size = 16

all_ct_probs = []
for i in tqdm(range(0, X_cond.shape[0], chuck_size)):
    x_cond = X_cond_resc_ij[i:i+chuck_size]
    fwd_out_orig = generate_with_condition(
        x_cond.detach().cpu().numpy(),
        forward_model,
        num_time_steps,
        solver_kwargs,
        ct_le,
        num_samples=350,
        noise=None,
        n_scatter_feats=6,
    )
    ct_probs = fwd_out_orig["ct_probs"]
    all_ct_probs.append(ct_probs)
all_ct_probs = np.concatenate(all_ct_probs, axis=1).mean(0)
resc_df_ij = pd.DataFrame(all_ct_probs, columns=classes)
resc_df_ij

### Query Forward Model with Rescaled Protocol (j -> i).

In [ ]:
num_time_steps = 10
solver_kwargs = {"method": "euler", "atol": 1e-5, "rtol": 1e-5}
chuck_size = 16

all_ct_probs = []
for i in tqdm(range(0, X_cond.shape[0], chuck_size)):
    x_cond = X_cond_resc_ji[i:i+chuck_size]
    fwd_out_orig = generate_with_condition(
        x_cond.detach().cpu().numpy(),
        forward_model,
        num_time_steps,
        solver_kwargs,
        ct_le,
        num_samples=350,
        noise=None,
        n_scatter_feats=6,
    )
    ct_probs = fwd_out_orig["ct_probs"]
    all_ct_probs.append(ct_probs)
all_ct_probs = np.concatenate(all_ct_probs, axis=1).mean(0)
resc_df_ji = pd.DataFrame(all_ct_probs, columns=classes)
resc_df_ji

### Concatenate and plot.

In [ ]:
# get values
y_orig = orig_df[target_ct]
y_resc_ij = resc_df_ij[target_ct]
y_resc_ji = resc_df_ji[target_ct]

# prepare plot
fig, ax = plt.subplots(1, 2, figsize=(4, 3))
fig.suptitle(f"Comparison Rescaling {target_ct}")

# ---- i to j ----
# set labels
ax[0].grid(True)
ax[0].set_xlabel("Original")
ax[0].set_ylabel("Rescaled i -> j")

# contructing linear space and plot x=y
linspace = np.linspace(0.0, .5, 100)
ax[0].plot(linspace, linspace, color="red")

# plot scatter of points
ax[0].scatter(y_orig, y_resc_ij)

# ---- j to i ----
# set labels
ax[1].grid(True)
ax[1].set_xlabel("Original")
ax[1].set_ylabel("Rescaled j -> i")

# contructing linear space and plot x=y
linspace = np.linspace(0.0, .5, 100)
ax[1].plot(linspace, linspace, color="red")

# plot scatter of points
ax[1].scatter(y_orig, y_resc_ji)

# display figure
fig.tight_layout()
fig.show()

### Check that Rescaling Makes Sense.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

print("R² i->j:", r2_score(y_orig, y_resc_ij))
print("MAE i->j:", mean_absolute_error(y_orig, y_resc_ij))

print("R² i->j:", r2_score(y_orig, y_resc_ji))
print("MAE i->j:", mean_absolute_error(y_orig, y_resc_ji))

# Also check if the error correlates with input value
input_vals = X_cond[:, protocol_columns.index("um171_[nm]")]
plt.figure(figsize=(3, 3))
plt.scatter(input_vals.cpu(), y_orig - y_resc_ij)
plt.xlabel("um171_[nm]"); plt.ylabel("Δ target CT prob")
plt.axhline(0, color='grey', ls='--')


plt.figure(figsize=(3, 3))
plt.scatter(input_vals.cpu(), y_orig - y_resc_ji)
plt.xlabel("um729_[nm]"); plt.ylabel("Δ target CT prob")
plt.axhline(0, color='grey', ls='--')

### Visualize Cell Type Composition.

In [ ]:
# Compute mean and std over conditions (axis=0)
orig_mean = orig_df.mean(axis=0)
orig_std  = orig_df.std(axis=0)

resc_mean_ij = resc_df_ij.mean(axis=0)
resc_std_ij  = resc_df_ij.std(axis=0)

resc_mean_ji = resc_df_ji.mean(axis=0)
resc_std_ji  = resc_df_ji.std(axis=0)

# Build a means DataFrame and an errors DataFrame for grouped bars
means = pd.DataFrame({'Original': orig_mean, 'Rescaled i->j': resc_mean_ij, 'Rescaled j->i': resc_mean_ji})
errors = pd.DataFrame({'Original': orig_std, 'Rescaled i->j': resc_std_ij, 'Rescaled j->i': resc_std_ij})

# Plot with error bars
ax = means.plot(kind='bar', yerr=errors, figsize=(12, 6), capsize=4, rot=45)
ax.set_ylabel('Probability')
ax.set_title('Cell-type composition: Original vs Rescaled (mean ± σ across conditions)')
ax.legend()
plt.tight_layout()
plt.show()
